# ML Assignment 2 - Model Training and Evaluation

This notebook trains the five classifiers named in the assignment on the same handwritten-digits dataset, calculates the six required evaluation metrics, and saves the files used by the Streamlit app.

In [1]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIRECTORY = Path.cwd()
PROJECT_ROOT = CURRENT_DIRECTORY.parent if CURRENT_DIRECTORY.name == "model" else CURRENT_DIRECTORY
MODEL_DIRECTORY = PROJECT_ROOT / "model"

if str(MODEL_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(MODEL_DIRECTORY))

from train_models import (
    RANDOM_STATE,
    TEST_SIZE,
    build_models,
    load_dataset,
    main,
)

print("Project root:", PROJECT_ROOT)
print("Random state:", RANDOM_STATE)
print("Test size:", TEST_SIZE)

Project root: /mnt/data/ML_Assignment_2_Digits
Random state: 42
Test size: 0.2


## 1. Load and inspect the dataset

In [2]:
X, y, dataset_info = load_dataset()

print("Dataset:", dataset_info["name"])
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Number of classes:", y.nunique())
print("Missing feature values:", int(X.isna().sum().sum()))
print("Class distribution:")
display(y.value_counts().sort_index().rename("count").to_frame())

Dataset: Optical Recognition of Handwritten Digits
Feature matrix shape: (1797, 64)
Target shape: (1797,)
Number of classes: 10
Missing feature values: 0
Class distribution:


,count
target,
0,178
1,182
2,177
3,183
4,181
5,182
6,181
7,179
8,174


## 2. Confirm the model list

In [3]:
models = build_models()
for number, (model_name, model) in enumerate(models.items(), start=1):
    print(f"{number}. {model_name}: {model}")

1. Logistic Regression: Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=3000, random_state=42))])
2. Decision Tree: DecisionTreeClassifier(random_state=42)
3. kNN: Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', KNeighborsClassifier(weights='distance'))])
4. Naive Bayes: Pipeline(steps=[('scaler', MinMaxScaler()),
                ('classifier', MultinomialNB(alpha=0.1))])
5. Random Forest (Ensemble): RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)


## 3. Train, evaluate, and save the five models

The training script uses an 80:20 stratified split. Precision, Recall, and F1 use weighted averaging. AUC uses weighted one-vs-rest averaging.

In [4]:
main()

Training and evaluating five models...



Logistic Regression             Accuracy=0.9722  AUC=0.9991  F1=0.9722  MCC=0.9692
Decision Tree                   Accuracy=0.8250  AUC=0.9028  F1=0.8237  MCC=0.8057
kNN                             Accuracy=0.9667  AUC=0.9950  F1=0.9664  MCC=0.9631
Naive Bayes                     Accuracy=0.8889  AUC=0.9911  F1=0.8881  MCC=0.8769


Random Forest (Ensemble)        Accuracy=0.9694  AUC=0.9992  F1=0.9692  MCC=0.9662

Saved artifacts:
- /mnt/data/ML_Assignment_2_Digits/test_data.csv
- /mnt/data/ML_Assignment_2_Digits/model/metrics.csv
- /mnt/data/ML_Assignment_2_Digits/model/metadata.json
- 5 serialized model files in /mnt/data/ML_Assignment_2_Digits/model

Best model: Logistic Regression


## 4. Comparison table

In [5]:
metrics = pd.read_csv(MODEL_DIRECTORY / "metrics.csv")
metric_columns = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
display(metrics.style.format({column: "{:.4f}" for column in metric_columns}))

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9722,0.9991,0.9724,0.9722,0.9722,0.9692
1,Decision Tree,0.8250,0.9028,0.8241,0.8250,0.8237,0.8057
2,kNN,0.9667,0.9950,0.9675,0.9667,0.9664,0.9631
3,Naive Bayes,0.8889,0.9911,0.8908,0.8889,0.8881,0.8769
4,Random Forest (Ensemble),0.9694,0.9992,0.9701,0.9694,0.9692,0.9662


## 5. Model observations

In [6]:
winner = metrics.sort_values(
    by=["F1", "Accuracy", "MCC", "AUC"], ascending=False
).iloc[0]["ML Model Name"]

print("Overall winner among the five models:", winner)
print()
print("Decision Tree has the lowest score and shows the high variance of a single tree.")
print("Random Forest improves substantially by averaging many decision trees.")
print("Logistic Regression and kNN benefit from scaled features.")
print("Naive Bayes is very fast, but the pixel-independence assumption limits its accuracy.")

Overall winner among the five models: Logistic Regression

Decision Tree has the lowest score and shows the high variance of a single tree.
Random Forest improves substantially by averaging many decision trees.
Logistic Regression and kNN benefit from scaled features.
Naive Bayes is very fast, but the pixel-independence assumption limits its accuracy.
